Cell 1: Setup & Load Gold

In [0]:
# MAGIC %md
# MAGIC # ⚡ EV Site Selection — NB04: Dynamic ROI Model
# MAGIC **Layer:** Gold (Enriched) | **Objective:** คำนวณ Payback Period แยกตาม Zone และ Flag ทำเล Golden Gap

from pyspark.sql.functions import col, when, lit, round as spark_round, acos, cos, sin, radians

DATABASE_NAME = "ev_site_selection"
spark.sql(f"USE {DATABASE_NAME}")

df_gold = spark.table("gold_grid_scored")
print(f"✅ Loaded {df_gold.count():,} records from gold_grid_scored")


✅ Loaded 18,492 records from gold_grid_scored


Cell 2: Financial Assumptions

In [0]:
# MAGIC %md
# MAGIC ## Cell 2: Financial Assumptions

CAPEX             = 1_500_000.0  # เงินลงทุนเริ่มต้น (บาท)
BASE_OPEX         =   120_000.0  # ค่าดำเนินการรายปี (บาท/ปี)
CBD_OPEX_PREMIUM  =         1.20 # ค่าใช้จ่ายเพิ่ม CBD (+20%)
URBAN_GROWTH_RATE =         0.05 # อัตราการเติบโตรายได้ Urban (5%/ปี)
PROFIT_PER_CAR    =        75.0  # กำไรต่อการชาร์จ 1 คัน (บาท)
ANALYSIS_YEARS    =            5 # ระยะเวลาวิเคราะห์ (ปี)

print("✅ Financial Assumptions Defined")


✅ Financial Assumptions Defined


Cell 3: Zonal Classification (Distance-based)

In [0]:
# MAGIC %md
# MAGIC ## Cell 3: Zonal Classification
# MAGIC CBD = ระยะ < 10 กม. จากใจกลาง กทม. | URBAN = นอกนั้น

BKK_CENTER_LAT = 13.7649
BKK_CENTER_LON = 100.5383

df_zoned = df_gold.withColumn("dist_to_center_km",
    acos(
        sin(radians(lit(BKK_CENTER_LAT))) * sin(radians(col("grid_lat"))) +
        cos(radians(lit(BKK_CENTER_LAT))) * cos(radians(col("grid_lat"))) *
        cos(radians(col("grid_lon")) - radians(lit(BKK_CENTER_LON)))
    ) * lit(6371.0)
).withColumn("zone_type",
    when(col("dist_to_center_km") < 10.0, "CBD").otherwise("URBAN")
).withColumn("annual_opex",
    when(col("zone_type") == "CBD", lit(BASE_OPEX * CBD_OPEX_PREMIUM))
    .otherwise(lit(BASE_OPEX))
)

print("✅ Zonal Logic Applied")
display(df_zoned.groupBy("zone_type").count())

✅ Zonal Logic Applied


zone_type,count
URBAN,17349
CBD,1143


Cell 4: Payback Period Calculation 

In [0]:
# MAGIC %md
# MAGIC ## Cell 4: Payback Period Calculation
# MAGIC จำลองกระแสเงินสด 5 ปี — CBD รายได้คงที่, URBAN รายได้โต 5%/ปี

df_roi = df_zoned \
    .withColumn("est_cars_per_day", spark_round((col("norm_demand") * 30) + 10, 0)) \
    .withColumn("year_1_revenue",   col("est_cars_per_day") * PROFIT_PER_CAR * 365)

cumulative_cash = lit(-CAPEX)
payback_expr    = lit(99.0)  # ค่า default ถ้าไม่คืนทุนใน 5 ปี

for year in range(1, ANALYSIS_YEARS + 1):
    yearly_rev = when(col("zone_type") == "CBD", col("year_1_revenue")) \
                .otherwise(col("year_1_revenue") * ((1 + URBAN_GROWTH_RATE) ** (year - 1)))
    yearly_net      = yearly_rev - col("annual_opex")
    cumulative_cash = cumulative_cash + yearly_net
    payback_expr    = when(
        (cumulative_cash > 0) & (payback_expr == 99.0),
        lit(year - 1) + ((cumulative_cash - yearly_net) * -1 / yearly_net)
    ).otherwise(payback_expr)

df_roi = df_roi \
    .withColumn("payback_years", spark_round(payback_expr, 2)) \
    .withColumn("is_viable",     when(col("payback_years") <= 4.0, True).otherwise(False)) \
    .withColumn("is_flood_safe", when(col("norm_risk") == 0.0,     True).otherwise(False)) \
    .withColumn("is_golden",     when(col("is_viable") & col("is_flood_safe"), True).otherwise(False))

print("✅ Payback Period Calculated")
display(df_roi.groupBy("zone_type", "is_golden").count())

✅ Payback Period Calculated


zone_type,is_golden,count
URBAN,false,15723
URBAN,true,1626
CBD,false,517
CBD,true,626


Cell 5: Save Enriched Gold Table

In [0]:
# MAGIC %md
# MAGIC ## Cell 5: Save Enriched Gold Table

df_roi.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_grid_scored")

print("📦 gold_grid_scored updated with ROI columns")

display(
    df_roi.filter(col("is_golden") == True)
          .select("grid_lat", "grid_lon", "zone_type", "est_cars_per_day", "payback_years")
          .orderBy("payback_years")
          .limit(10)
)

📦 gold_grid_scored updated with ROI columns


grid_lat,grid_lon,zone_type,est_cars_per_day,payback_years
13.774,100.6376,URBAN,40.0,1.51
13.7695,100.6376,URBAN,40.0,1.51
13.765,100.6478,URBAN,36.0,1.69
13.7605,100.6478,URBAN,36.0,1.69
13.7605,100.6427,URBAN,36.0,1.69
13.765,100.6427,URBAN,36.0,1.69
13.7785,100.6376,URBAN,35.0,1.75
13.774,100.6325,URBAN,35.0,1.75
13.7695,100.6325,URBAN,35.0,1.75
13.783,100.6376,URBAN,35.0,1.75
